# Chess dataset EDA

How often does the teacher model return a well-formed continuation (right
format, right length) for the chess-analogue of the numbers dataset, and how
often is that continuation also *legal* chess? All logic lives in
[`chess_eda_lib.py`](chess_eda_lib.py) - this notebook is config + calls, same
convention as [`eval.ipynb`](eval.ipynb).

Pipeline: sample `size` prompts from `ChessDatasetPromptSet` (a random 3-8
half-move opening prefix from the lichess-openings book, asking the model to
continue with up to `answer_count` more half-moves), run each completion
through `chess_dataset.get_reject_reasons` (format + count check only) to get
`is_valid`, and additionally replay every format-valid completion on a board
with `python-chess` to get `is_legal`.

The legality check lives **only in this notebook** (`ceda.check_legal`), for
exploratory analysis. The actual dataset-generation pipeline
(`sl.datasets.chess_dataset.get_reject_reasons`, used by
`cfgs/run_cfg.py`/`run_experiment.sh`) deliberately stays format-only,
mirroring how the numbers dataset filter checks format/range/count and
nothing deeper - so this notebook's `is_legal` column is diagnostic, not part
of the training-data filter.

In [ ]:
%load_ext autoreload
%autoreload 2

import chess_eda_lib as ceda
from sl.datasets.services import ChessDatasetPromptSet
from sl.llm.data_models import Model, SampleCfg

## Config

Swap `MODEL` to point at the base model you want to check (a local vLLM
open-source model, or a cheap OpenAI model for a quick smoke test of the
pipeline itself before burning GPU time on Qwen).

In [ ]:
# --- Open-source model via local vLLM (needs the `open_models` extra + a GPU) ---
#MODEL = Model(id="qwen2.5:7b", type="ollama")
MODEL = Model(id="gemma3:4b", type="ollama")

SIZE = 100  # number of prompts to sample

prompt_set = ChessDatasetPromptSet(
    size=SIZE,
    seed=0,
    example_min_count=3,
    example_max_count=9,
    answer_count=10,
)
sample_cfg = SampleCfg(temperature=1)

## Sample + score

In [ ]:
df = await ceda.sample_and_check(MODEL, prompt_set, sample_cfg)
print(df.shape)
df.head()

In [ ]:
ceda.reject_reason_counts(df)

In [ ]:
df["prompt"].head(5).tolist()

## Valid rate + reject-reason breakdown

In [ ]:
ceda.summary(df)

In [ ]:
ceda.plot_reject_reasons(df)

In [ ]:
ceda.plot_valid_rate_by_seed_length(df)

## Legal rate (of format-valid completions)

Format-valid doesn't mean legal - a completion can parse as a clean list of
SAN-looking tokens while still describing an illegal game. `is_legal` is
`None` for rows that already failed the format check (legality is undefined
for those).

In [ ]:
ceda.plot_legal_rate_by_seed_length(df)

## Look at a sample of rejected completions

In [ ]:
df[~df.is_valid][["prompt", "completion", "reject_reasons"]].sample(
    min(10, (~df.is_valid).sum())
)

## Look at a sample of format-valid but illegal completions

In [ ]:
illegal_df = df[df.is_valid & ~df.is_legal.astype("boolean")]
illegal_df[["prompt", "completion"]].sample(min(10, len(illegal_df)))